# Classical Experiment 5 — Line Spectra, the Ritz Principle, and Bohr's Atom
*(Kirchhoff–Bunsen 1859; Balmer 1885; Rydberg 1890; Ritz 1908; Bohr 1913 — Weinberg §1.2)*
**What the experiments showed (and where classical physics broke)**

- **Each element emits its own discrete line spectrum** (Kirchhoff–Bunsen): hydrogen's visible lines fit by
  Balmer (1885), $\lambda = b\,n^2/(n^2-4)$, $n = 3,4,\ldots$
- **Rydberg (1890):** $\nu = R_{\rm H}\left(\frac{1}{n^2} - \frac{1}{m^2}\right)$; the series *converge to limits*
  ($364.6$ nm for Balmer).
- **Ritz combination principle (1908):** every line frequency is a *difference of terms* — the spectrum is the
  pairwise difference set of a small list of numbers $T_n$.
- **Contradiction with the Rutherford atom:** an accelerating electron must radiate a continuum (previous
  notebook). Discrete, combinable lines are impossible classically.
- **Bohr (1913):** stationary states $E_n = -R_{\rm H}hc/n^2$ + $h\nu = E_m - E_n$ at transitions + quantization
  $m_{\rm e}vr = n\hbar$ (Eqs. 1.2.3–1.2.4), reproducing Rydberg exactly and predicting new series (Lyman,
  Paschen — found 1908/1909 in the infrared, recycled by Bohr as confirmation).


In [ ]:
%matplotlib inline
import numpy as np, matplotlib.pyplot as plt, sys
sys.path.insert(0, '..')
from weinberg_constants import h, hbar, e, me, c, Ryd

eV = 1.602176634e-12
m_p = 1.6726e-24
mu_H = me/(1 + me/m_p)
R_H  = mu_H*e**4/(2*hbar**2)/h          # Rydberg frequency, Hz
print(f"R_H = {R_H:.4e} Hz   = {R_H/c*100:.5f} x 10^7 m^-1   (accepted 1.09678)")


## 1. The Balmer series in the visible
Bar under a wavelength axis; each line at its measured color, converging to the series limit at 364.6 nm.
Classical expectation (from the collapse notebook): a continuous wash, not four sharp lines.

In [ ]:
nm = np.arange(380, 760, 0.5)
continuous = np.exp(-((nm-550)/140)**2)     # what a continuum would look like
fig, ax = plt.subplots(figsize=(11, 3.2))
ax.imshow(np.tile(nm, (8, 1)), extent=[380, 760, 0, 1], aspect='auto',
          cmap='gray_r', vmin=0, vmax=1, alpha=0.0)   # placeholder axis
ax.plot(nm, continuous*0.45, lw=1.5, color='tab:red', ls='--', label='classical continuum (chirp)')
names = ['H-alpha','H-beta','H-gamma','H-delta']
colors = ['#c33', '#35a', '#35a', '#46b']
for k, n in enumerate(range(3, 7)):
    lam = 1/(R_H*(1/4 - 1/n**2))/1e-7       # nm
    ax.axvline(lam, color=colors[k], lw=2.5)
    ax.text(lam, 0.55 + 0.12*(k%2), f'{names[k]}\n{lam:.1f}', ha='center', fontsize=8)
lam_lim = 1/(R_H/4)/1e-7
ax.axvline(lam_lim, color='k', lw=1, ls=':')
ax.text(lam_lim, 0.9, f'  series limit {lam_lim:.1f} nm', fontsize=8)
ax.set_xlim(380, 760); ax.set_ylim(0, 1.15); ax.set_yticks([])
ax.set_xlabel('wavelength (nm)')
ax.set_title('Hydrogen emission, visible (Balmer 1885)')
ax.legend(fontsize=8, loc='upper right')
plt.tight_layout(); plt.show()


## 2. Term diagram: the Ritz combination principle made visible
Horizontal lines = Bohr terms $E_n = -13.6/n^2$ eV; arrows = observed lines, labeled with wavelength.
Every drawn frequency is a *difference of two terms* — the whole spectrum from 5 numbers.

In [ ]:
E_n = lambda n: -13.6/n**2          # eV
levels = np.arange(1, 6)
series = {1: ('Lyman', '#a4c'), 2: ('Balmer', '#c33'), 3: ('Paschen', '#393')}

fig, ax = plt.subplots(figsize=(8, 6))
for n in levels:
    ax.hlines(E_n(n), 0, 1, color='k', lw=1.5)
    ax.text(1.02, E_n(n), f'n={n}  ({E_n(n):.2f} eV)', va='center', fontsize=9)
for m_start, (sname, cc) in series.items():
    for m in range(m_start+1, 7):
        lam_nm = 1e7/(109678*(1/m_start**2 - 1/m**2))
        if lam_nm > 3000: continue
        x = 0.15 + 0.14*(m - m_start - 1) + 0.05*m_start
        ax.annotate('', xy=(x, E_n(m_start)), xytext=(x, E_n(m)),
                    arrowprops=dict(arrowstyle='->', color=cc, lw=1.5))
        ax.text(x+0.012, (E_n(m)+E_n(m_start))/2, f'{lam_nm:.0f}', fontsize=6.5, color=cc)
ax.text(0.06, -14.6, series[1][0], color=series[1][1], fontsize=10)
ax.text(0.38, -8.4, series[2][0], color=series[2][1], fontsize=10)
ax.text(0.70, -3.4, series[3][0], color=series[3][1], fontsize=10)
ax.set_xlim(-0.05, 1.45); ax.set_ylim(-15.5, -0.2)
ax.set_xticks([]); ax.set_ylabel('energy (eV)')
ax.set_title('Bohr term diagram of hydrogen (wavelengths in nm)')
plt.tight_layout(); plt.show()


## 3. The spectrum is the difference set of the terms
Matrix of all frequencies $\nu_{mn} = R_{\rm H}(1/n^2 - 1/m^2)$ — the Ritz combination principle: if lines at
$\nu_1$ and $\nu_2$ exist, a line at $|\nu_1 - \nu_2|$ exists. This structural fact (1908) was the template Bohr
filled in with physics (1913).

In [ ]:
grid = np.zeros((7, 7))
for n in range(1, 7):
    for m in range(n+1, 8):
        grid[n-1, m-1] = R_H*(1/n**2 - 1/m**2)/1e12   # x10^12 Hz
fig, ax = plt.subplots(figsize=(6.5, 5))
im = ax.imshow(grid, cmap='viridis')
for n in range(7):
    for m in range(7):
        if grid[n, m] > 0:
            ax.text(m, n, f'{grid[n,m]:.2f}', ha='center', va='center', fontsize=7.5,
                    color='w' if grid[n,m] < 2.5 else 'k')
ax.set_xticks(range(7), [f'm={m+1}' for m in range(7)])
ax.set_yticks(range(7), [f'n={n+1}' for n in range(7)])
ax.set_title(r'Ritz grid: $\nu_{mn}=R_H(1/n^2-1/m^2)$  ($10^{12}$ Hz)')
fig.colorbar(im, label=r'$\nu$ ($10^{12}$ Hz)')
plt.tight_layout(); plt.show()


## 4. Reduced mass: the He$^+$ vs H fingerprint
$E_n = -Z^2\mu e^4/(2\hbar^2 n^2)$ with $\mu = m_{\rm e}m_{\rm N}/(m_{\rm e}+m_{\rm N})$: He$^+$ lines sit at
nearly $4\times$ the H frequencies — the tiny offset ($4.00163$, not $4$) measures the nuclear mass ratio and
identified "hydrogen-like" lines in stellar spectra (Pickering series).

In [ ]:
m_He = 6.6464e-24
mu_He = me/(1 + me/m_He)
def E_lev(Z, n, mu): return -Ryd*(mu/me)*Z**2/n**2/eV    # eV

fig, ax = plt.subplots(figsize=(7.5, 5.5))
for n in range(1, 6):
    for xoff, (Z, mu, lab, cc) in [(0.05, (1, mu_H, 'H', 'tab:blue')), (0.55, (2, mu_He, 'He$^+$', 'tab:orange'))]:
        ax.hlines(E_lev(Z, n, mu), xoff, xoff+0.4, color=cc, lw=1.5)
        ax.text(xoff+0.42, E_lev(Z, n, mu), f'{E_lev(Z, n, mu):.2f}', fontsize=7.5, color=cc, va='center')
ax.text(0.25, 2.5, 'hydrogen', color='tab:blue', ha='center', fontsize=11)
ax.text(0.75, 2.5, r'He$^+$ ($Z^2$ scale, $\mu$ shift)', color='tab:orange', ha='center', fontsize=11)
ax.set_xlim(0, 1); ax.set_xticks([]); ax.set_ylabel('energy (eV)')
ax.set_title('Two Rydberg ladders: same physics, $Z^2$ and $\\mu$ shifts')
plt.tight_layout(); plt.show()

ratio = E_lev(2, 1, mu_He)/E_lev(1, 1, mu_H)
print(f"He+ / H ground-state ratio = {ratio:.5f}   [Weinberg: 4.00163]")


## My summary
- Line spectra + Ritz's difference rule say: the atom has a small set of privileged energies, and light is emitted
  in jumps between them.
- Bohr's three postulates (stationary states, $h\nu = E_m - E_n$, $L = n\hbar$) turn that arithmetic into physics
  and reproduce Rydberg with $R_{\rm H} = \mu e^4/2\hbar^2h$ — a *prediction* of $R$ in terms of $e$, $m$, $h$.
- The $\mu$-shift ($4.00163$) shows the theory already contains enough structure to weigh nuclei spectroscopically.
- What Bohr does **not** explain: line *intensities*, elliptic orbits (Sommerfeld), or why states are stationary at
  all — that wait for Schrödinger/Heisenberg (§1.3–1.4).

## Gaps
- Balmer found his formula by fitting $n^2/(n^2-4)$ — confirm numerically it is Rydberg with $n=2$ terminal state.
- Where exactly does Sommerfeld's general quantization $\oint p\,dq = nh$ (Eq. 1.2.8) extend this? (defer to §1.2)
- The Ritz grid above is finite — why does hydrogen have infinitely many bound terms accumulating at 0?